In [1]:
import numpy as np
import pandas as pd
from ast import literal_eval
from pathlib import Path

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
best_system_ids = [10, 50, 51]
systems_cleaned[systems_cleaned['system_id'].isin(best_system_ids)]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.12,BSk,12,...,True,True,True,True,True,cis family thin-film,thin_film,PVDAQ General,5893,2007
8,50,NREL x-Si 6,"Golden, CO",7,39.7420,-105.1727,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8455,1995
9,51,NREL x-Si 7,"Golden, CO",7,39.7416,-105.1734,1994.7,6.00,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8032,1995


In [3]:
param_grid_xgb = {
    'num_leaves': (7, 15, 31),
    'max_depth': (5, 7, 10),
    'learning_rate': (0.1, 0.2),
    'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0),
}

param_grid_lgb = {
    'num_leaves': (7, 15, 31),
    'max_depth': (-1, 5, 7),
    'learning_rate': (0.1, 0.2),
    'n_estimators': (100,),  # wanted more, but there is literally no time!
    'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0),
}

## XGB Reading

In [4]:
def formatter(boost_type: str, system_id: int):
    if boost_type == 'xgboost':
        read_path = Path(f'./xgboost_results/{system_id}_None.csv')
    elif boost_type == 'lightgbm':
        read_path = Path(f'./lightgbm_results/{system_id}_None.csv')
    elif boost_type == 'lightgbm_alt':
        read_path = Path(f'./lightgbm_results/{system_id}_None_null_drops.csv')
    else:
        raise ValueError('Invalid boost_type!')
    results_df = pd.read_csv(read_path)
    # transpose for future use
    results_df = results_df.transpose()
    # grab current columns
    current_columns = results_df.columns
    # grab means and standard deviations in aggregate
    results_df['per_model_mean'] = results_df[current_columns].mean(axis=1)
    results_df['per_model_std'] = results_df[current_columns].std(axis=1)
    # separate params
    results_df.index.name = 'params'
    results_df = results_df.reset_index()
    #string to tuple components
    results_df['params'] = results_df['params'].apply(literal_eval)
    if boost_type == 'xgboost':
        params_list = ('num_leaves', 'max_depth', 'learning_rate',
                       'subsample', 'colsample_bytree')
    elif boost_type == 'lightgbm' or boost_type == 'lightgbm_alt':
        params_list = ('num_leaves', 'max_depth', 'learning_rate',
                       'n_estimators',
                       'subsample', 'colsample_bytree')
    results_df.loc[:, params_list] = results_df['params'].to_list()
    results_df = results_df.set_index('params')
    # rearrange columns
    ordinary_columns = [col_name for col_name in results_df.columns
        if (col_name not in params_list)
            and (col_name not in ('per_model_mean', 'per_model_std'))]
    results_df = results_df[[*params_list, *ordinary_columns, 'per_model_mean', 'per_model_std']]
    # rename
    results_df = results_df.rename(columns = {
        j: f'Day {j}' for j in range(len(ordinary_columns))
    })
    return results_df


In [5]:
params_list_xgb = ('num_leaves', 'max_depth', 'learning_rate',
                   'subsample', 'colsample_bytree')
params_list_lgb = ('num_leaves', 'max_depth', 'learning_rate',
                   'n_estimators',
                   'subsample', 'colsample_bytree')

In [6]:
xgb_10 = formatter('xgboost', 10)
lgb_10 = formatter('lightgbm', 10)
xgb_50 = formatter('xgboost', 50)
lgb_50 = formatter('lightgbm', 50)
xgb_51 = formatter('xgboost', 51)
lgb_51 = formatter('lightgbm', 51)

In [ ]:
xgb_50_alt = formatter('lightgbm_alt', 50)

In [ ]:
xgb_10.head()

,num_leaves,max_depth,learning_rate,subsample,colsample_bytree,Day 0,Day 1,Day 2,Day 3,Day 4,...,Day 314,Day 315,Day 316,Day 317,Day 318,Day 319,Day 320,Day 321,per_model_mean,per_model_std
params,,,,,,,,,,,,,,,,,,,,,
"(7, 5, 0.1, 0.8, 0.8)",7,5,0.1,0.8,0.8,0.069389,0.047776,0.005776,0.107149,0.079663,...,0.008794,0.029069,0.006462,0.022185,0.001711,0.056223,0.040127,0.163227,0.039327,0.035720
"(7, 5, 0.1, 0.8, 1.0)",7,5,0.1,0.8,1.0,0.065556,0.039819,0.003784,0.093358,0.066090,...,0.007930,0.026034,0.005956,0.022244,0.001180,0.057373,0.039143,0.158771,0.039268,0.036570
"(7, 5, 0.1, 1.0, 0.8)",7,5,0.1,1.0,0.8,0.070462,0.045263,0.002651,0.108239,0.081525,...,0.008210,0.027501,0.005848,0.022756,0.001895,0.055593,0.038738,0.158266,0.039469,0.035651
"(7, 5, 0.1, 1.0, 1.0)",7,5,0.1,1.0,1.0,0.074105,0.046778,0.006129,0.092201,0.095792,...,0.009141,0.030612,0.006189,0.021941,0.001248,0.057176,0.040933,0.156780,0.039837,0.036783
"(7, 5, 0.2, 0.8, 0.8)",7,5,0.2,0.8,0.8,0.075905,0.044304,0.007564,0.095446,0.065155,...,0.010151,0.031808,0.007173,0.020518,0.001382,0.064805,0.040159,0.159945,0.040269,0.038056


In [ ]:
xgb_50_alt

,num_leaves,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,Day 0,Day 1,Day 2,Day 3,...,Day 221,Day 222,Day 223,Day 224,Day 225,Day 226,Day 227,Day 228,per_model_mean,per_model_std
params,,,,,,,,,,,,,,,,,,,,,
"(7, -1, 0.1, 100, 0.8, 0.8)",7,-1,0.1,100,0.8,0.8,0.427370,0.168686,1.647572,4.677221,...,0.364469,1.488126,0.626004,0.289061,0.780915,0.608330,0.767425,0.174716,1.059503,1.028745
"(7, -1, 0.1, 100, 0.8, 1.0)",7,-1,0.1,100,0.8,1.0,0.431901,0.116943,1.821641,4.884454,...,0.347942,1.349169,0.853493,0.320783,0.857494,0.499259,0.609175,0.152078,1.039526,1.040958
"(7, -1, 0.1, 100, 1.0, 0.8)",7,-1,0.1,100,1.0,0.8,0.427370,0.168686,1.647572,4.677221,...,0.364469,1.488126,0.626004,0.289061,0.780915,0.608330,0.767425,0.174716,1.059503,1.028745
"(7, -1, 0.1, 100, 1.0, 1.0)",7,-1,0.1,100,1.0,1.0,0.431901,0.116943,1.821641,4.884454,...,0.347942,1.349169,0.853493,0.320783,0.857494,0.499259,0.609175,0.152078,1.039526,1.040958
"(7, -1, 0.2, 100, 0.8, 0.8)",7,-1,0.2,100,0.8,0.8,0.410901,0.185466,1.658044,4.421899,...,0.417636,1.276509,0.757198,0.240650,0.869192,0.564114,0.563692,0.180954,1.053652,1.038446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"(31, 7, 0.1, 100, 1.0, 1.0)",31,7,0.1,100,1.0,1.0,0.346942,0.124640,2.029548,2.574232,...,0.404739,1.298653,0.789598,0.269658,0.863833,0.408470,0.634175,0.229427,0.966520,0.987291
"(31, 7, 0.2, 100, 0.8, 0.8)",31,7,0.2,100,0.8,0.8,0.206978,0.161692,1.815943,3.430443,...,0.465142,1.112986,0.623842,0.292614,0.880902,0.464128,0.460360,0.199314,1.007977,1.044211
"(31, 7, 0.2, 100, 0.8, 1.0)",31,7,0.2,100,0.8,1.0,0.435215,0.098952,1.870304,2.919210,...,0.362176,1.164840,0.924661,0.348285,0.850510,0.618402,0.567944,0.236146,1.001493,1.025293


In [7]:
def find_differences(df: pd.DataFrame, boost_type: str, system_id: int):
    print(f'System {system_id}, boosting with {boost_type}')
    print(df[['Day 0', 'Day 200', 'per_model_mean', 'per_model_std']].describe())
    print('')
    for param in ('num_leaves', 'max_depth', 'learning_rate',
                   'subsample', 'colsample_bytree'):
        print(param)
        print(f'{param} Effect on mean error')
        print(df.groupby([param])['per_model_mean'].agg(['mean', 'std']))
        print(f'{param} Effect on st.dev. of error')
        print(df.groupby([param])['per_model_std'].agg(['mean', 'std']))
        print('')

In [8]:
find_differences(xgb_10, 'xgboost', 10)

System 10, boosting with xgboost
           Day 0    Day 200  per_model_mean  per_model_std
count  72.000000  72.000000       72.000000      72.000000
mean    0.075219   0.001345        0.039700       0.038292
std     0.006356   0.000422        0.000528       0.001525
min     0.061251   0.000723        0.038613       0.035651
25%     0.070732   0.001131        0.039344       0.037114
50%     0.074734   0.001255        0.039599       0.038078
75%     0.078768   0.001480        0.039901       0.039549
max     0.089872   0.002638        0.041072       0.041966

num_leaves
num_leaves Effect on mean error
                mean       std
num_leaves                    
7           0.039693  0.000351
15          0.039617  0.000452
31          0.039789  0.000720
num_leaves Effect on st.dev. of error
                mean       std
num_leaves                    
7           0.036880  0.001019
15          0.038397  0.001046
31          0.039597  0.001084

max_depth
max_depth Effect on mean error
  

In [ ]:
find_differences(xgb_50, 'xgboost', 50)

System 50, boosting with xgboost
           Day 0    Day 200  per_model_mean  per_model_std
count  72.000000  72.000000       72.000000      72.000000
mean    2.112319   1.120120        1.087325       1.293370
std     0.143435   0.178939        0.019397       0.066907
min     1.907717   0.851448        1.041629       1.180843
25%     2.000972   1.022189        1.079065       1.239314
50%     2.104476   1.077457        1.087092       1.306191
75%     2.181336   1.152533        1.097531       1.333912
max     2.590329   1.711589        1.125097       1.556026

num_leaves
num_leaves Effect on mean error
                mean       std
num_leaves                    
7           1.099094  0.014866
15          1.087998  0.014757
31          1.074884  0.020402
num_leaves Effect on st.dev. of error
                mean       std
num_leaves                    
7           1.260146  0.059106
15          1.332086  0.052052
31          1.287878  0.069776

max_depth
max_depth Effect on mean error
  

In [ ]:
find_differences(xgb_51, 'xgboost', 51)

System 51, boosting with xgboost
           Day 0    Day 200  per_model_mean  per_model_std
count  72.000000  72.000000       72.000000      72.000000
mean    1.949127   0.677316        1.155438       1.094696
std     0.170168   0.175861        0.031226       0.055623
min     1.577054   0.479889        1.086789       1.009615
25%     1.822817   0.582659        1.136545       1.047786
50%     1.937434   0.655446        1.163299       1.091492
75%     2.100139   0.722430        1.177997       1.126449
max     2.277717   1.508313        1.205494       1.257296

num_leaves
num_leaves Effect on mean error
                mean       std
num_leaves                    
7           1.178785  0.008889
15          1.151066  0.019499
31          1.136463  0.039829
num_leaves Effect on st.dev. of error
                mean       std
num_leaves                    
7           1.080025  0.034311
15          1.097019  0.034189
31          1.107043  0.082555

max_depth
max_depth Effect on mean error
  

In [ ]:
find_differences(lgb_10, 'lightgbm', 10)

System 10, boosting with lightgbm
validation_day_index      Day 0    Day 321  per_model_mean  per_model_std
count                 72.000000  72.000000       72.000000      72.000000
mean                   0.082808   0.167300        0.039897       0.039389
std                    0.008679   0.009759        0.000571       0.001863
min                    0.067765   0.147990        0.039082       0.036451
25%                    0.077364   0.162305        0.039576       0.038308
50%                    0.082809   0.166665        0.039752       0.039093
75%                    0.087264   0.172780        0.040052       0.040121
max                    0.102716   0.187380        0.041505       0.045140

num_leaves
num_leaves Effect on mean error
                mean       std
num_leaves                    
7           0.039712  0.000110
15          0.039907  0.000463
31          0.040071  0.000844
num_leaves Effect on st.dev. of error
                mean       std
num_leaves                    
7

In [ ]:
find_differences()

In [ ]:
def find_differences_to_file(df: pd.DataFrame, boost_type: str, system_id: int,
                             file_path: Path):
    with open(file_path, 'w') as writer:
        print(f'System {system_id}, gradient boosting with {boost_type}', file=writer)
        print(df[['Day 0', 'Day 200', 'per_model_mean', 'per_model_std']].describe(),
              file=writer)
        print('', file=writer)
        for param in ('num_leaves', 'max_depth', 'learning_rate',
                   'subsample', 'colsample_bytree'):
            print(param, file=writer)
            print(f'{param} Effect on mean error', file=writer)
            print(df.groupby([param])['per_model_mean'].agg(['mean', 'std']),
                  file=writer)
            print(f'{param} Effect on st.dev. of error', file=writer)
            print(df.groupby([param])['per_model_std'].agg(['mean', 'std']),
                  file=writer)
            print('', file=writer)

In [ ]:
best_system_ids

[10, 50, 51]

In [ ]:
for system_id in best_system_ids:
    for boost_type in ('xgboost', 'lightgbm'):
        my_data = formatter(boost_type, system_id)
        find_differences_to_file(
            my_data, boost_type, system_id,
            Path(f'./{boost_type}_results/{boost_type}_{system_id}_summary.txt')
        )

In [9]:
num_bests = 1
for boost_type in ('xgboost', 'lightgbm'):
    for system_id in best_system_ids:
        print(f'System {system_id}, boosting {boost_type}')
        my_data = formatter(boost_type, system_id)
        mean_best = my_data['per_model_mean'].min()
        best_data = my_data[my_data['per_model_mean'] <1.0001 * mean_best]
        print(best_data[['per_model_mean', 'per_model_std']])
        print('')
        num_bests = max(num_bests, best_data.shape[0])
num_bests

System 10, boosting xgboost
                         per_model_mean  per_model_std
params                                                
(31, 7, 0.1, 0.8, 0.8)         0.038613        0.03805
(31, 10, 0.1, 0.8, 0.8)        0.038613        0.03805

System 50, boosting xgboost
                         per_model_mean  per_model_std
params                                                
(31, 7, 0.1, 0.8, 0.8)         1.041629       1.206276
(31, 10, 0.1, 0.8, 0.8)        1.041629       1.206276

System 51, boosting xgboost
                        per_model_mean  per_model_std
params                                               
(31, 5, 0.1, 1.0, 0.8)        1.086789       1.011577

System 10, boosting lightgbm
                              per_model_mean  per_model_std
params                                                     
(31, -1, 0.1, 100, 0.8, 0.8)        0.039082       0.038583
(31, -1, 0.1, 100, 1.0, 0.8)        0.039082       0.038583

System 50, boosting lightgbm
            

2

In [ ]:
candidates_10 = xgb_10[(xgb_10['num_leaves'] == 31)
       & (xgb_10['learning_rate'] == 0.1)
       & (xgb_10['colsample_bytree'] == 0.8)].copy(deep=True)
candidates_10 = candidates_10[['per_model_mean', 'per_model_std']]
candidates_10['per_model_mean'] = candidates_10['per_model_mean'] / systems_cleaned.at[3, 'dc_capacity_kW']
candidates_10['per_model_std'] = candidates_10['per_model_std'] / systems_cleaned.at[3, 'dc_capacity_kW']
candidates_10 = candidates_10.rename(columns = {'per_model_mean': 'norm_mean_10', 'per_model_std': 'norm_std_10'})

In [ ]:
candidates_50 = xgb_50[(xgb_50['num_leaves'] == 31)
       & (xgb_50['learning_rate'] == 0.1)
       & (xgb_50['colsample_bytree'] == 0.8)].copy(deep=True)
candidates_50 = candidates_50[['per_model_mean', 'per_model_std']]
candidates_50['per_model_mean'] = candidates_50['per_model_mean'] / systems_cleaned.at[8, 'dc_capacity_kW']
candidates_50['per_model_std'] = candidates_50['per_model_std'] / systems_cleaned.at[8, 'dc_capacity_kW']
candidates_50 = candidates_50.rename(columns = {'per_model_mean': 'norm_mean_50', 'per_model_std': 'norm_std_50'})

In [ ]:
candidates_51 = xgb_51[(xgb_51['num_leaves'] == 31)
       & (xgb_51['learning_rate'] == 0.1)
       & (xgb_51['colsample_bytree'] == 0.8)].copy(deep=True)
candidates_51 = candidates_51[['per_model_mean', 'per_model_std']]
candidates_51['per_model_mean'] = candidates_51['per_model_mean'] / systems_cleaned.at[9, 'dc_capacity_kW']
candidates_51['per_model_std'] = candidates_51['per_model_std'] / systems_cleaned.at[9, 'dc_capacity_kW']
candidates_51 = candidates_51.rename(columns = {'per_model_mean': 'norm_mean_51', 'per_model_std': 'norm_std_51'})

In [ ]:
candidates = pd.concat([candidates_10, candidates_50, candidates_51], axis=1)
candidates

,norm_mean_10,norm_std_10,norm_mean_50,norm_std_50,norm_mean_51,norm_std_51
params,,,,,,
"(31, 5, 0.1, 0.8, 0.8)",0.035163,0.034559,0.174060,0.207891,0.183164,0.169114
"(31, 5, 0.1, 1.0, 0.8)",0.034689,0.034472,0.175644,0.201705,0.181131,0.168269
"(31, 7, 0.1, 0.8, 0.8)",0.034476,0.033921,0.173605,0.200704,0.183356,0.172253
"(31, 7, 0.1, 1.0, 0.8)",0.034852,0.034793,0.178329,0.209312,0.182315,0.172272
"(31, 10, 0.1, 0.8, 0.8)",0.034476,0.033921,0.173605,0.200704,0.183356,0.172253
"(31, 10, 0.1, 1.0, 0.8)",0.034819,0.034788,0.178329,0.209312,0.182476,0.173104


In [ ]:
candidates['norm_mean_10'].max() / candidates['norm_mean_10'].min()

np.float64(1.0199205431735936)

In [ ]:
candidates.iloc[1, 0] / candidates.iloc[2, 0]

np.float64(1.0061877840283615)

In [ ]:
candidates.iloc[1, 2] / candidates.iloc[2, 2]

np.float64(1.011746220435831)

In [ ]:
candidates.iloc[2, 4] / candidates.iloc[1, 4]

np.float64(1.0122814519379013)

In [ ]:
candidates['norm_mean_50'].max() / candidates['norm_mean_50'].min()

np.float64(1.0272118316236565)

### OKay, so the System 51 optimizer does better on SYstems 10/50 then reverse.

In [ ]:
candidates['norm_mean_51'].max() / candidates['norm_mean_51'].min()

np.float64(1.0122814519379013)

In [ ]:
candidates_51 = xgb_51[(xgb_51['num_leaves'] == 31)
       & (xgb_51['learning_rate'] == 0.1)
       & (xgb_51['colsample_bytree'] == 0.8)][['per_model_mean', 'per_model_std']]
candidates_51 = candidates_10.rename(columns = ['mean_50', 'std_50'])